# 01 · Panorama 2022 → 2026

Primeira olhada nos dados processados (`just repro`). **Exploratório**: nada aqui é teste
confirmatório. O que virar afirmação vai antes para `docs/hipoteses.md`.

Métrica principal: `pt_2p`, a fatia do PT entre os dois maiores (PT / (PT + adversário)).
Comparações entre pleitos usam o painel por **AMC**, que tem as mesmas fronteiras em 2022 e 2026.

In [ ]:
import altair as alt
import polars as pl

from kingsrace import paths
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import SERIES, TINTA, escala_divergente

alt.data_transformers.disable_max_rows()

P = paths.PROCESSED
municipios = pl.read_parquet(P / "municipios.parquet")
painel = pl.read_parquet(P / "painel_municipio.parquet")
amc = pl.read_parquet(P / "painel_amc.parquet")
votos = pl.read_parquet(P / "votos_municipio.parquet")

## 1. Resultado nacional

Inclui o exterior. Percentuais sobre os votos válidos; candidatos com menos de 1% omitidos.

In [ ]:
nacional = (
    votos.filter(pl.col("papel").is_in(["pt", "adversario", "outro"]))
    .group_by("ano", "turno", "nm_votavel")
    .agg(pl.col("votos").sum())
    .with_columns(
        (100 * pl.col("votos") / pl.col("votos").sum().over("ano", "turno"))
        .round(2)
        .alias("% válidos")
    )
    .filter(pl.col("% válidos") >= 1)
    .sort("ano", "turno", "votos", descending=[False, False, True])
)
nacional

In [ ]:
comparecimento = (
    painel.group_by("ano", "turno")
    .agg(pl.col("aptos", "comparecimento", "abstencoes", "votos_pt", "votos_adv").sum())
    .with_columns(
        (100 * pl.col("abstencoes") / pl.col("aptos")).round(2).alias("% abstenção"),
        (100 * pl.col("votos_pt") / (pl.col("votos_pt") + pl.col("votos_adv")))
        .round(2)
        .alias("PT % 2 maiores"),
    )
    .sort("ano", "turno")
)
comparecimento  # só municípios do Brasil (sem exterior)

## 2. Como o voto no PT se distribui entre os municípios

Quantos municípios em cada faixa de `pt_2p` no 1º turno. A distribuição de 2022 tem dois
picos (redutos do PT e do adversário); a pergunta é o que mudou em 2026.

In [ ]:
dist = painel.filter(pl.col("turno") == 1).select(
    pl.col("ano").cast(pl.String), (100 * pl.col("pt_2p")).alias("pt_2p_pct")
)
cores_ano = alt.Scale(domain=["2022", "2026"], range=SERIES[:2])
base = (
    alt.Chart(dist)
    .transform_bin("faixa", "pt_2p_pct", bin=alt.Bin(step=2.5))
    .transform_aggregate(municipios="count()", groupby=["ano", "faixa"])
)
linhas = base.mark_line(interpolate="step-after").encode(
    x=alt.X("faixa:Q", title="PT entre os dois maiores (%)", scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("municipios:Q", title="Municípios"),
    color=alt.Color("ano:N", scale=cores_ano, title="1º turno"),
    tooltip=[
        alt.Tooltip("ano:N", title="Ano"),
        alt.Tooltip("faixa:Q", title="Faixa a partir de (%)"),
        alt.Tooltip("municipios:Q", title="Municípios"),
    ],
)
meio = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(x=alt.datum(50))
(linhas + meio).properties(
    width=640, height=260, title="Distribuição dos municípios por voto no PT (1º turno)"
)

## 3. A queda de Lula e o tamanho do município

Variação de `pt_2p` entre o 1º turno de 2022 e o de 2026, em pontos percentuais, por AMC.
Negativo = o PT perdeu espaço para o adversário. Cada ponto é um município; a linha é uma
suavização LOESS. Um painel por região.

In [ ]:
uf_regiao = dict(municipios.select("uf", "regiao").unique().iter_rows())
nomes_amc = municipios.group_by("amc_id").agg(
    pl.col("nome").sort().str.join(" + "), pl.col("uf").first().alias("uf_amc")
)
delta = (
    amc.filter(pl.col("turno") == 1)
    .pivot(on="ano", index=["amc_id", "uf"], values=["pt_2p", "aptos"])
    .join(nomes_amc, on="amc_id")
    .with_columns(
        delta_pp=100 * (pl.col("pt_2p_2026") - pl.col("pt_2p_2022")),
        pt_2022=100 * pl.col("pt_2p_2022"),
        regiao=pl.col("uf").replace_strict(uf_regiao),
    )
)
nac = comparecimento.filter(pl.col("turno") == 1).select("ano", "PT % 2 maiores")
print("Variação nacional do PT entre os dois maiores (pp):", nac["PT % 2 maiores"].diff()[1])
delta.select("delta_pp").describe()

In [ ]:
ordem_regioes = ["N", "NE", "CO", "SE", "S"]
eixo_x = alt.X(
    "aptos_2026:Q",
    title="Eleitores aptos em 2026 (escala log)",
    scale=alt.Scale(type="log"),
    axis=alt.Axis(values=[1e3, 1e4, 1e5, 1e6, 1e7], format="~s"),
)
pontos = (
    alt.Chart()
    .mark_circle(size=12, opacity=0.25, color=SERIES[0])
    .encode(
        x=eixo_x,
        y=alt.Y("delta_pp:Q", title="Variação do PT entre os 2 maiores (pp)"),
        tooltip=[
            alt.Tooltip("nome:N", title="Município"),
            alt.Tooltip("uf:N", title="UF"),
            alt.Tooltip("aptos_2026:Q", title="Aptos 2026", format=",d"),
            alt.Tooltip("pt_2022:Q", title="PT 2022 (%)", format=".1f"),
            alt.Tooltip("delta_pp:Q", title="Variação (pp)", format="+.1f"),
        ],
    )
)
suave = (
    alt.Chart()
    .transform_calculate(log_aptos="log(datum.aptos_2026) / LN10")
    .transform_loess("log_aptos", "delta_pp", bandwidth=0.5)
    .transform_calculate(aptos_2026="pow(10, datum.log_aptos)")
    .mark_line(color=TINTA["primaria"], strokeWidth=2)
    .encode(x=eixo_x, y="delta_pp:Q")
)
zero = alt.Chart().mark_rule(color=TINTA["apagada"]).encode(y=alt.datum(0))
alt.layer(zero, pontos, suave, data=delta).properties(width=210, height=220).facet(
    facet=alt.Facet("regiao:N", sort=ordem_regioes, title=None), columns=3
).properties(title="Variação do PT entre os dois maiores, 1º turno 2022 → 2026, por município")

### Média por faixa de tamanho × região

Média da variação ponderada pelo eleitorado de 2026. A cor segue a convenção do projeto:
**vermelho = rumo ao PT, azul = rumo ao adversário**, cinza = sem variação (escala de −10 a +10 pp).

In [ ]:
faixas = [5_000, 10_000, 20_000, 50_000, 100_000, 500_000]
rotulos = [
    "< 5 mil",
    "5–10 mil",
    "10–20 mil",
    "20–50 mil",
    "50–100 mil",
    "100–500 mil",
    "≥ 500 mil",
]
por_faixa = (
    delta.with_columns(faixa=pl.col("aptos_2026").bin_intervals(faixas, labels=rotulos))
    .group_by("regiao", "faixa")
    .agg(
        ((pl.col("delta_pp") * pl.col("aptos_2026")).sum() / pl.col("aptos_2026").sum()).alias(
            "delta_medio"
        ),
        pl.len().alias("municipios"),
    )
    .with_columns(pl.col("faixa").cast(pl.String))
)
calor = alt.Chart(por_faixa).encode(
    x=alt.X("faixa:N", sort=rotulos, title="Eleitores aptos em 2026", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("regiao:N", sort=ordem_regioes, title=None),
)
celulas = calor.mark_rect(stroke="#fcfcfb", strokeWidth=2, cornerRadius=4).encode(
    color=alt.Color("delta_medio:Q", scale=escala_divergente(10), title="Variação média (pp)"),
    tooltip=[
        alt.Tooltip("regiao:N", title="Região"),
        alt.Tooltip("faixa:N", title="Faixa"),
        alt.Tooltip("delta_medio:Q", title="Variação média (pp)", format="+.1f"),
        alt.Tooltip("municipios:Q", title="Municípios"),
    ],
)
rotulos_txt = calor.mark_text(fontSize=11).encode(
    text=alt.Text("delta_medio:Q", format="+.1f"),
    # texto claro nas células escuras (|variação| > 6 pp)
    color=alt.condition(
        "abs(datum.delta_medio) > 6", alt.value("#ffffff"), alt.value(TINTA["primaria"])
    ),
)
(celulas + rotulos_txt).properties(
    width=560,
    height=200,
    title="Variação média do PT (pp) por tamanho e região, 1º turno 2022 → 2026",
)

## Antes de concluir qualquer coisa

- **Composição:** tamanho, região, renda e agro andam juntos; os painéis por região só
  controlam o primeiro corte. Próximo passo (M1a): covariáveis do Censo e efeitos fixos.
- **Teto:** onde o PT tinha 85% há mais espaço para cair; repetir com `pt_2p_logit`.
- **Ruído:** municípios minúsculos espalham os pontos; olhe a linha, não as nuvens.
- **Falácia ecológica:** são municípios, não eleitores.